# 01 — Data Cleaning
**London Crime Analysis (2020–2024)**

Loads raw Met Police street-level crime CSVs, cleans and merges into one analysis-ready dataset.

In [4]:
import pandas as pd
import numpy as np
import os
import glob

RAW_DIR  = '../data/raw/'
OUT_PATH = '../data/processed/london_crime_clean.csv'
os.makedirs('../data/processed', exist_ok=True)
print('Paths OK')

Paths OK


## 1. Load All CSV Files

data.police.uk gives one CSV per month. We stack all of them.

In [5]:
all_files = glob.glob(os.path.join(RAW_DIR, '**/*.csv'), recursive=True)
print(f'Found {len(all_files)} CSV files')

frames = []
for f in all_files:
    try:
        frames.append(pd.read_csv(f, low_memory=False))
    except Exception as e:
        print(f'Could not read {f}: {e}')

df = pd.concat(frames, ignore_index=True)
print(f'Total rows loaded: {len(df):,}')
print('Columns:', df.columns.tolist())
df.head(3)

Found 36 CSV files
Total rows loaded: 3,415,820
Columns: ['Crime ID', 'Month', 'Reported by', 'Falls within', 'Longitude', 'Latitude', 'Location', 'LSOA code', 'LSOA name', 'Crime type', 'Last outcome category', 'Context']


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,7c85d5925620a55fd890ea37dd748a34d2c77773f5d740...,2023-04,Metropolitan Police Service,Metropolitan Police Service,-0.254186,50.834677,On or near Kingsland Close,E01031372,Adur 004E,Vehicle crime,Investigation complete; no suspect identified,NaN
1,43ccf73d9d380e4fcd128d7384d49ccf8d5ed54887a940...,2023-04,Metropolitan Police Service,Metropolitan Police Service,-0.559343,50.852819,On or near School Lane,E01031392,Arun 001C,Violence and sexual offences,Investigation complete; no suspect identified,NaN
2,dd453aa29e0058e3fd98c507bf9429b10d654b653571f7...,2023-04,Metropolitan Police Service,Metropolitan Police Service,-0.405658,50.865992,On or near Southview Road,E01031425,Arun 002C,Violence and sexual offences,Investigation complete; no suspect identified,NaN


## 2. Inspect

In [6]:
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())
print('\nCrime types:')
print(df['Crime type'].value_counts())

Shape: (3415820, 12)

Missing values:
Crime ID                  695743
Month                          0
Reported by                    0
Falls within                   0
Longitude                  16313
Latitude                   16313
Location                       0
LSOA code                  16314
LSOA name                  16314
Crime type                     0
Last outcome category     695743
Context                  3415820
dtype: int64

Crime types:
Crime type
Violence and sexual offences    795167
Anti-social behaviour           695743
Other theft                     336616
Vehicle crime                   279763
Theft from the person           256888
Shoplifting                     235470
Public order                    171302
Criminal damage and arson       166923
Burglary                        153913
Drugs                           130957
Robbery                          96779
Bicycle theft                    44833
Other crime                      35882
Possession of weapons

## 3. Clean

In [7]:
# Standardise column names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)

# Parse month (YYYY-MM) and extract time parts
df['month'] = pd.to_datetime(df['month'], format='%Y-%m')
df['year']       = df['month'].dt.year
df['month_num']  = df['month'].dt.month
df['month_name'] = df['month'].dt.strftime('%B')
df['year_month'] = df['month'].dt.to_period('M').astype(str)

# Filter 2020–2024
df = df[(df['month'] >= '2023-04-01') & (df['month'] <= '2026-03-31')].copy()
print(f'Rows after date filter: {len(df):,}')
print('Range:', df['month'].min(), '→', df['month'].max())

Rows after date filter: 3,415,820
Range: 2023-04-01 00:00:00 → 2026-03-01 00:00:00


In [8]:
# Drop rows missing critical fields
before = len(df)
df.dropna(subset=['crime_type', 'lsoa_name'], inplace=True)
print(f'Dropped {before - len(df):,} rows with missing crime_type or lsoa_name')

# Extract borough from lsoa_name e.g. 'Hackney 001A' → 'Hackney'
df['borough'] = df['lsoa_name'].str.rsplit(' ', n=2).str[0]

# Clean crime type casing
df['crime_type'] = df['crime_type'].str.strip().str.title()

print('Sample boroughs:', df['borough'].value_counts().head(8).index.tolist())

Dropped 16,314 rows with missing crime_type or lsoa_name
Sample boroughs: ['Westminster', 'Camden', 'Newham', 'Southwark', 'Lambeth', 'Tower', 'Croydon', 'Ealing']


In [9]:
# Keep relevant columns only
keep = ['month', 'year', 'month_num', 'month_name', 'year_month',
        'crime_type', 'borough', 'lsoa_code', 'lsoa_name',
        'latitude', 'longitude', 'last_outcome_category']
keep = [c for c in keep if c in df.columns]
df = df[keep].copy()

print(f'Final shape: {df.shape}')
df.head()

Final shape: (3399506, 12)


,month,year,month_num,month_name,year_month,crime_type,borough,lsoa_code,lsoa_name,latitude,longitude,last_outcome_category
0,2023-04-01,2023,4,April,2023-04,Vehicle Crime,Adur,E01031372,Adur 004E,50.834677,-0.254186,Investigation complete; no suspect identified
1,2023-04-01,2023,4,April,2023-04,Violence And Sexual Offences,Arun,E01031392,Arun 001C,50.852819,-0.559343,Investigation complete; no suspect identified
2,2023-04-01,2023,4,April,2023-04,Violence And Sexual Offences,Arun,E01031425,Arun 002C,50.865992,-0.405658,Investigation complete; no suspect identified
3,2023-04-01,2023,4,April,2023-04,Violence And Sexual Offences,Arun,E01031389,Arun 005B,50.821606,-0.481480,Status update unavailable
4,2023-04-01,2023,4,April,2023-04,Violence And Sexual Offences,Arun,E01031469,Arun 009F,50.816593,-0.543948,Status update unavailable


## 4. Save

In [10]:
df.to_csv(OUT_PATH, index=False)
print(f'✅ Saved {len(df):,} rows → {OUT_PATH}')
print('\nFinal crime types:')
print(df['crime_type'].value_counts())

✅ Saved 3,399,506 rows → ../data/processed/london_crime_clean.csv

Final crime types:
crime_type
Violence And Sexual Offences    791006
Anti-Social Behaviour           695743
Other Theft                     333999
Vehicle Crime                   277719
Theft From The Person           254888
Shoplifting                     234657
Public Order                    170232
Criminal Damage And Arson       165958
Burglary                        153174
Drugs                           130342
Robbery                          96144
Bicycle Theft                    44528
Other Crime                      35612
Possession Of Weapons            15504
Name: count, dtype: int64
